# Train BanglaBERT without answer leakage

This notebook fine-tunes `csebuetnlp/banglabert` to classify a question into one of the 13 Physics chapters. It uses only question text and chapter labels from `physics_questions_234.json`. No reference answers, answer spans, generated explanations, or textbook answer passages are loaded.

The web app does not use this classifier to write answers. The user selects a chapter, retrieval is hard-filtered to that chapter, and Qwen writes a grounded response only from retrieved chunks.

In [ ]:
import sys
assert 'google.colab' in sys.modules, 'Training is intentionally restricted to Google Colab'
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
!pip -q install 'transformers==4.46.3' 'datasets==3.2.0' 'accelerate==1.2.1' 'sentencepiece>=0.2.0'

## Upload the question-only dataset

Upload `dataset/physics_questions_234.json`. The notebook explicitly rejects rows containing answer-bearing fields.

In [ ]:
from collections import Counter, defaultdict
import json
from pathlib import Path
from google.colab import files
uploaded = files.upload()
dataset_name = next((name for name in uploaded if name == 'physics_questions_234.json'), None)
assert dataset_name, 'Upload physics_questions_234.json'
document = json.loads(Path(dataset_name).read_text(encoding='utf-8'))
examples = document['questions']
for row in examples:
    forbidden = {'answer', 'answers', 'answer_start', 'direct_answer', 'reference_answer', 'display_answer', 'explanation', 'context'}
    assert not (forbidden & row.keys()), f"Answer-bearing field found in {row.get('id')}"
    assert set(('question', 'chapter_id')) <= row.keys()
assert len(examples) == 234
assert Counter(int(row['chapter_id']) for row in examples) == {chapter: 18 for chapter in range(1, 14)}
print('Validated 234 question-only examples; no answers were loaded')

In [ ]:
from datasets import Dataset, DatasetDict
by_chapter = defaultdict(list)
for row in examples:
    by_chapter[int(row['chapter_id'])].append(row)
train_rows, validation_rows = [], []
for chapter_id in range(1, 14):
    rows = sorted(by_chapter[chapter_id], key=lambda row: row['id'])
    train_rows.extend(rows[:15])
    validation_rows.extend(rows[15:])
def model_row(row):
    return {'text': row['question'], 'label': int(row['chapter_id']) - 1}
raw = DatasetDict({
    'train': Dataset.from_list([model_row(row) for row in train_rows]),
    'validation': Dataset.from_list([model_row(row) for row in validation_rows]),
})
raw

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
MODEL_CHECKPOINT = 'csebuetnlp/banglabert'
OUTPUT_DIR = '/content/drive/MyDrive/pathshongi-banglabert-physics-chapters'
id2label = {index: f'chapter_{index + 1}' for index in range(13)}
label2id = {label: index for index, label in id2label.items()}
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=13, id2label=id2label, label2id=label2id)
tokenized = raw.map(lambda batch: tokenizer(batch['text'], truncation=True, max_length=128), batched=True)
collator = DataCollatorWithPadding(tokenizer)

In [ ]:
from transformers import Trainer, TrainingArguments
def metrics(result):
    predictions = np.argmax(result.predictions, axis=-1)
    return {'accuracy': float((predictions == result.label_ids).mean())}
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=8, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='accuracy',
    report_to='none', seed=42,
)
trainer = Trainer(model=model, args=training_args, train_dataset=tokenized['train'], eval_dataset=tokenized['validation'], tokenizer=tokenizer, data_collator=collator, compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
Path(OUTPUT_DIR, 'training_metadata.json').write_text(json.dumps({
    'task': 'chapter-classification',
    'training_inputs': ['question', 'chapter_id'],
    'contains_answers': False,
    'labels': id2label,
}, indent=2), encoding='utf-8')

## Interpretation

Validation accuracy measures chapter classification only. It is not answer quality, and this checkpoint must never be presented as an answer generator. Evaluate grounded answers separately on questions whose wording and answers are absent from training data.